# AFNI: Task fMRI GLM Analysis

Complete pipeline using AFNI: motion correction -> smoothing -> GLM (3dDeconvolve) -> t-maps -> viewer. Uses ds000114 finger/foot/lips task fMRI.

## Prerequisites

- **Python 3.9+**
- **AFNI** installed and in your system PATH (https://afni.nimh.nih.gov/pub/dist/doc/htmldoc/background_install/main_toc.html)
- **datalad** for downloading the ds000114 dataset
- Basic understanding of task fMRI and the general linear model (GLM)

## Setup

### Data Download

This notebook uses the **ds000114** dataset (finger/foot/lips motor task).
Download it using datalad:

```bash
pip install datalad
datalad install https://github.com/OpenNeuroDatasets/ds000114
cd ds000114
datalad get sub-01/ses-test/func/sub-01_ses-test_task-fingerfootlips_bold.nii.gz
datalad get task-fingerfootlips_events.tsv
```

Make sure the `ds000114` directory is in your working directory (or adjust `DS114` below).

### AFNI Installation Check

AFNI should be in your system PATH. Verify with:
```bash
which afni
afni --version
```

In [ ]:
# Install required packages
!pip install nibabel nilearn matplotlib numpy pandas

In [ ]:
import subprocess, os, glob, nibabel as nib
import nilearn.plotting as nlp
import matplotlib.pyplot as plt

# Check that AFNI is in PATH (don't hardcode ~/abin)
afni_check = subprocess.run(['which', '3dvolreg'], capture_output=True, text=True)
if afni_check.returncode != 0:
    raise RuntimeError(
        'AFNI not found in PATH. Please install AFNI and ensure it is on your PATH.\n'
        'See: https://afni.nimh.nih.gov/pub/dist/doc/htmldoc/background_install/main_toc.html'
    )
print('AFNI found at:', afni_check.stdout.strip())

# Data paths (relative -- assumes ds000114 is in the working directory)
DS114 = 'ds000114'
FUNC = f'{DS114}/sub-01/ses-test/func/sub-01_ses-test_task-fingerfootlips_bold.nii.gz'
EVENTS = f'{DS114}/task-fingerfootlips_events.tsv'

# Output directory (portable, relative path)
OUT = 'output/afni_glm'
os.makedirs(OUT, exist_ok=True)

img = nib.load(FUNC)
print(f'fMRI shape: {img.shape}  (x, y, z, TRs)')
print(f'TR: {img.header.get_zooms()[3]:.2f}s')

In [ ]:
import pandas as pd, numpy as np

# Parse events file -> AFNI timing files (1 column per condition)
events = pd.read_csv(EVENTS, sep='\t')
print(events.head(10))
conditions = events['trial_type'].unique()
print('\nConditions:', conditions)

for cond in conditions:
    onsets = events[events['trial_type'] == cond]['onset'].values
    timing_file = f'{OUT}/timing_{cond}.1D'
    np.savetxt(timing_file, onsets.reshape(1, -1), fmt='%.2f')
    print(f'  {cond}: {len(onsets)} events -> {timing_file}')

In [ ]:
# Step 1: Despike + slice timing + motion correction
volreg_out = f'{OUT}/sub01_volreg.nii.gz'
result = subprocess.run([
    '3dvolreg', '-Fourier', '-twopass', '-zpad', '4',
    '-prefix', volreg_out, FUNC
], capture_output=True, text=True, cwd=OUT)
print('3dvolreg:', 'OK' if os.path.exists(volreg_out) else result.stderr[:200])

In [ ]:
# Step 2: Spatial smoothing (6mm FWHM)
smooth_out = f'{OUT}/sub01_smooth.nii.gz'
result = subprocess.run([
    '3dmerge', '-1blur_fwhm', '6.0', '-doall',
    '-prefix', smooth_out, volreg_out
], capture_output=True, text=True, cwd=OUT)
print('3dmerge smooth:', 'OK' if os.path.exists(smooth_out) else result.stderr[:200])

In [ ]:
# Step 3: GLM with 3dDeconvolve
# Use the actual condition names from the events file (case-sensitive!)
# The timing files are saved as timing_Finger.1D, timing_Foot.1D, timing_Lips.1D
stim_args = []
for i, cond in enumerate(conditions, 1):
    tf = f'{OUT}/timing_{cond}.1D'
    if os.path.exists(tf):
        stim_args += ['-stim_times', str(i), tf, 'GAM',
                      '-stim_label', str(i), cond]
    else:
        print(f'WARNING: timing file not found: {tf}')

glm_prefix = f'{OUT}/sub01_glm'
cmd = [
    '3dDeconvolve',
    '-input', smooth_out,
    '-polort', '2',
    '-num_stimts', str(len(conditions)),
] + stim_args + [
    '-tout', '-fout',
    '-bucket', glm_prefix,
    '-cbucket', f'{glm_prefix}_coef',
    '-overwrite'
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=OUT)
print('3dDeconvolve:', 'OK' if result.returncode == 0 else result.stderr[-500:])

In [ ]:
# Visualize t-statistic maps
import glob
bucket = glob.glob(f'{glm_prefix}+orig.BRIK') or glob.glob(f'{glm_prefix}.nii.gz')
if bucket:
    result = subprocess.run(['3dAFNItoNIFTI', '-prefix', f'{glm_prefix}.nii.gz',
                              f'{glm_prefix}+orig'], capture_output=True, text=True, cwd=OUT)
    stat_img = nib.load(f'{glm_prefix}.nii.gz')
    print('GLM output shape:', stat_img.shape, '(last dim = sub-bricks: betas + t-stats)')
    # Plot the finger t-stat (sub-brick index 1 typically)
    from nilearn.image import index_img
    tstat = index_img(f'{glm_prefix}.nii.gz', 1)
    nlp.plot_stat_map(tstat, threshold=3.0, title='Finger movement t-stat (3dDeconvolve)',
                      display_mode='z', cut_coords=5)
    plt.show()
else:
    print('GLM output not found -- check 3dDeconvolve output above for errors.')

## References

- Cox, R. W. (1996). AFNI: Software for analysis and visualization of functional magnetic resonance neuroimages. *Computers and Biomedical Research*, 29(3), 162-173. https://doi.org/10.1006/cbmr.1996.0014
- Cox, R. W., & Hyde, J. S. (1997). Software tools for analysis and visualization of fMRI data. *NMR in Biomedicine*, 10(4-5), 171-178. https://doi.org/10.1002/(SICI)1099-1492(199706/08)10:4/5<171::AID-NBM453>3.0.CO;2-L
- Gorgolewski, K. J., Auer, T., Calhoun, V. D., et al. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. *Scientific Data*, 3, 160044. https://doi.org/10.1038/sdata.2016.44
- ds000114 on OpenNeuro: https://openneuro.org/datasets/ds000114
- AFNI documentation: https://afni.nimh.nih.gov/pub/dist/doc/htmldoc/